# Profiling Applications
---

The purpose of this notebook is to teach the basic concept of profiling deep neural networks with multi-GPU. You will learn how to use the NVIDIA Tools Extension (NVTX) tag to annotate your code and the command to profile it and analyze it using the NVIDIA® Nsight™ Systems GUI. Please note that your result might vary based on the GPU architecture used for the profile. All results are based on `DGX A100.`

We will consider using the DDP code from the previous notebook (distributed data parallelism) with multi-GPU for learning purposes. The first step would be to use the NVIDIA® Nsight™ Systems command line interface (CLI) to profile our DDP code. The rule of thumb is to limit the profiling to the second or third epoch. The first epoch is meant to “warm up” the GPU, and the second and subsequent epochs would have the same workload. We will use the APIs `cudaProfilerStart` to set the profiling start point and `cudaProfilerStop` to mark the stop region within our code, as shown below. For this section, our profiling will be limited to just the `second epoch` throughout the optimization workload.

```python
...
scheduler = StepLR(optimizer, step_size=1, gamma=args.gamma)
for epoch in range(1, args.epochs + 1):
        # Start profiling from 2nd epoch
        if epoch == 2:
            torch.cuda.cudart().cudaProfilerStart()
        train(args, model, device, train_loader, optimizer, epoch)
        test(model, device, test_loader)
        scheduler.step()
        # Stop profiling at the end of 2nd epoch
        if epoch == 2:
            torch.cuda.cudart().cudaProfilerStop()
...
```

Below is a sample command to profile an application.

```python

!nsys profile --trace cuda,osrt \
--capture-range cudaProfilerApi \
--output ../reports/baseline \
--force-overwrite true \
--gpu-metrics-devices=all \
--gpu-metrics-frequency=5000 \
python3 <application-name>.py

```


**Profiling command definitions:**

- `nsys`: command to call Nsight Systems CLI 
- `profile`: command to instruct profiling
- `--trace`: flag that represents tracing of APIs 
- `cuda,osrt`: APIs to be traced 
- `--capture-range`: the flag that indicates the start and stop profiling range using `cudaProfilerStart()` and `CudaProfilerStop()` APIs. 
- `--output`: flag that indicates the output of the profile  
- `../reports/baseline`: directory to save the profile output as `baseline.qdrep` or `baseline.nsys-rep` 
- `--gpu-metrics-devices=all`: fetches GPU metrics like GPC clock frequency, SYS clock frequency, SM instruction, and SM Wrap occupancy. The value can also be set to `cuda-visible`
- `--gpu-metrics-frequency=5000`:  fetches GPU metrics for 5kHz (i.e., 5ms)
- `--force-overwrite`: flag that denotes overwrite existing output file. It is usually set as `true` or `false`
- `python3`: command to run the DNN code in linux/ubuntu OS
- `<application-name>.py`: path to the DNN python code
A detailed user guide on Nsight Systems CLI profiling commands is presented [here](https://docs.nvidia.com/drive/drive-os-5.2.3.0L/nsight-systems/pdf/UserGuide.pdf). 

Let's run a simple [DDP program](../source_code/ddp_baseline.py) with 4 GPUs and time the execution duration. The program uses the resnet18 model and the CIFAR10 dataset.


In [ ]:
!cd ../source_code && torchrun --nproc_per_node=4 --nnodes=1 --standalone --master_addr="localhost" --master_port=1234 ddp_baseline.py

**Likely output on A100:**

```python
...
Local Rank: 2, Epoch: 23, Training ...
Local Rank: 1, Epoch: 23, Training ...
Local Rank: 3, Epoch: 23, Training ...
Local Rank: 0, Epoch: 23, Training ...
Local Rank: 2, Epoch: 24, Training ...
Local Rank: 1, Epoch: 24, Training ...
Local Rank: 3, Epoch: 24, Training ...
Local Rank: 0, Epoch: 24, Training ...
Total elapsed time: 100.95 seconds
Total elapsed time: 101.43 seconds
Total elapsed time: 101.46 seconds
Total elapsed time: 101.50 seconds

```

It takes 101.50 seconds to execute the 25 epochs in the training program.

### Basic Performance Issues to Address 

There are three basic performance issues that are common to DNN applications, and these include:
- Data loading
- Data transfer (H to D with `cudaMemcpyAsync`)
- Absence of Tensor Core usage

We will demonstrate how to identify these basic issues by profiling a DDP application. But before we proceed, we need to learn about the [NVIDIA Tools Extension](https://nvtx.readthedocs.io/en/latest/index.html) (NVTX) annotation library to annotate our application. The annotation helps to easily identify which part of our code has bottlenecks and hotspots to address.  The NVTX library provides functions for annotating events, code ranges, and resources in your application. The NVTX APIs offer additional information for NVIDIA’s tools while incurring almost no overhead when the tool is not attached to the application. A sample code to add NVTX annotations looks like the following:   

```python
    from torch.cuda import nvtx
    
    ....

    nvtx.range_push("annotation string")

     #lines of code to annotate

    nvtx.range_pop(); 
    
    .....

```

Let's examine our simple [DDP application](../source_code/ddp-baseline_nvtx.py). The inclusion of `nvtx` in the `--trace` flag enables the trace of the code where `nvtx` annotation is specified. Please run the cell below to profile.

In [ ]:
!cd ../source_code && nsys profile --trace cuda,osrt,nvtx \
--capture-range cudaProfilerApi \
--gpu-metrics-devices=cuda-visible \
--gpu-metrics-frequency=5000 \
--cuda-flush-interval=0 \
--output ../reports/baseline_nvtx \
--force-overwrite true \
torchrun --nproc_per_node=4 --nnodes=1 --standalone --master_addr="localhost" --master_port=1234 ddp-baseline_nvtx.py

When the profiling is done, we will inspect the report using Nsight Systems' graphical user interface (GUI). Download the [baseline_nvtx.nsys-rep](../reports/baseline_nvtx.nsys-rep) file at `../reports/baseline_nvtx.nsys-rep` and view it in the NVIDIA Nsight Systems GUI.

Each of the three issues shows up as a distinctive pattern in the timeline.  We'll look at the symptom in the trace here, but the **fixes themselves are covered in the intro lab** (which built up the same diagnostic skills on a single-GPU workload) — there's no need to re-learn them.

**Data loading inspection**

The NVTX `data_load` ranges show large gaps between forward and backward passes — every GPU stalls each step while the host fetches and decodes the next batch.  These idle gaps repeat at the data-load cadence:

<center><img src="images/baseline_dataloading.png" width="800px" height="800px" alt-text="data loading gaps"/></center>

**Fix**: the standard DataLoader knobs — `num_workers > 0`, `pin_memory=True`, `persistent_workers=True`, optionally `prefetch_factor=4`.  See the [DataLoader Knobs to Know](intro-pytorch-profiler.ipynb#DataLoader-Knobs-to-Know) table in the intro lab for the full reference.

**Data transfer inspection**

To analyze host↔device transfers, expand the NVIDIA CUDA® HW row of each GPU (click the small triangle), select the `Memory` row, and right-click → `Show in Events View`:

<center><img src="images/baseline-show-event-view.png" width="800px" height="800px" alt-text="event view"/></center>

Sort by Duration descending; the longest copy operations float to the top.  Right-click → `Zoom to Select on Timeline` to find them in context:

<center><img src="images/baseline-pageable.png" width="800px" height="800px" alt-text="pageable copies"/></center>

The teal highlights link each `Memcpy HtoD` to the `cudaMemcpyAsync` CPU call that initiated it.  Two symptoms of pageable-memory copies:

- The copies block the CPU thread instead of running asynchronously (the CPU call's duration is comparable to the GPU copy's duration, rather than a few microseconds for a non-blocking launch).
- Individual copies are large and slow — in this trace, the longest single HtoD copy is ~603 ms on GPU1.

<center><img src="images/baseline-cudamemcpyAsync.png" width=100% alt-text="cudaMemcpyAsync detail"/></center>

**Fix**: `pin_memory=True` on the DataLoader, paired with `.to(device, non_blocking=True)` on the training-loop side.  Both are covered in the [intro lab](intro-pytorch-profiler.ipynb).

**Tensor Core inspection**

The `--gpu-metrics-devices=cuda-visible` flag we passed to nsys captures SM-level metrics including how often Tensor Cores are active.  Click on any of the GPU rows (e.g. `GPU (000:47:00.0-NVIDIA A100-SXM4-80GB)`), expand `GPU Metrics [10kHz]`, then `SM Instructions`, and look at the `Tensor Active` row.  A baseline FP32 training run shows this row near zero — Tensor Cores aren't being used at all.

<center><img src="images/baseline-tensor-core.png" width="850px" height="850px" alt-text="Tensor Active row"/></center>

**Fix**: mixed-precision training via `torch.amp.autocast`.  See the [intro AMP lab](intro-amp.ipynb) — modern best practice is bfloat16 without a gradient scaler.  Once AMP is enabled, the `Tensor Active` row should light up during forward and backward kernels.

### Profile to Verify the Optimization

The next step is to profile again and verify if the [code changes](../source_code/ddp_optimize.py) address the bottlenecks. Please run the command in the cell below.

In [ ]:
!cd ../source_code && nsys profile --trace cuda,osrt,nvtx \
--capture-range cudaProfilerApi \
--gpu-metrics-devices=cuda-visible \
--gpu-metrics-frequency=5000 \
--cuda-flush-interval=0 \
--output ../reports/firstOptim \
--force-overwrite true \
torchrun --nproc_per_node=4 --nnodes=1 --standalone --master_addr="localhost" --master_port=1234 ddp_optimize.py

Download the report [firstOptim.nsys-rep](../reports/firstOptim.nsys-rep) and open it in the Nsight Systems GUI. Zoom in on the data loading and data transfer (HtoD) on the event view timeline. You will notice a reduction in the time spent on both tasks. The report below shows that the time taken to load data on both GPU0 and GPU2 has reduced significantly. 

<center><img src="images/opt-dataloading.png" width="850px" height="850px" alt-text="workflow"/></center>
<br/>





Before switching to pinned memory, the longest process (HtoD) on GPU1 (`process id: #7051`) took `~603,000` microseconds, and `431,262` microseconds on GPU2 (`process id:#4051`). After the optimization steps, the longest processes on both devices (`process id:6408`) are now noticeably lesser. 
<center><img src="images/opt-GPU1-pinned.png" width="850px" height="850px" alt-text="workflow"/></center>
<br/>

It is important to note that a process ID can be assigned to another process at the next application run. This was the case with process IDs `#7051 (HtoD)` and `#4051 (HtoD),` as they were assigned to a new process, `DtoD` on both devices (*as indicated in the report screenshot with green frames*)


<center><img src="images/opt-GPU2-pinned.png" width="850px" height="850px" alt-text="workflow"/></center>
<br/>




### Compare the Performance Before and After the Optimizations

Now that we have addressed some fundamental performance problems, let's time the [DDP application](../source_code/ddp_run_optimize_update.py). Please run the command in the cell below. 

In [ ]:
!cd ../source_code && torchrun --nproc_per_node=4 --nnodes=1 --standalone --master_addr="localhost" --master_port=1234 ddp_run_optimize_update.py

**Likely Output on DGX A100:**

```python
...
Local Rank: 2, Epoch: 23, Training ...
Local Rank: 1, Epoch: 23, Training ...
Local Rank: 3, Epoch: 23, Training ...
Local Rank: 0, Epoch: 23, Training ...
Local Rank: 2, Epoch: 24, Training ...
Local Rank: 0, Epoch: 24, Training ...
Local Rank: 3, Epoch: 24, Training ...
Local Rank: 1, Epoch: 24, Training ...
Total elapsed time: 61.28 seconds
Total elapsed time: 57.58 seconds
Total elapsed time: 62.31 seconds
Total elapsed time: 62.68 seconds
```

When we compare the time to run our baseline [DDP program code](../source_code/ddp_baseline.py) with the basic-level [optimization DDP code](../source_code/ddp_run_optimize.py), we see that the overall time has decreased, as shown in the table below.

|DDP code|Time|Speedup|
|--|--|--|
|Baseline|101s| - |
|Basic optimization|62s|~1.63x |

Now that we have learned basic multi-GPU profiling, let's continue to the next notebook to explore more features of the NVIDIA Nsight System and reveal further insight into our DDP program. Please click the link below.

## <center><div style="text-align:center; color:#FF0000; border:3px solid red;height:80px;"> <b><br/> [Next Notebook](nsys-trace.ipynb) </b> </div></center>




---

## Links and Resources


[NVIDIA Nsight Systems](https://developer.nvidia.com/nsight-systems)


**NOTE**: To be able to see the profiler output, please download the latest version of NVIDIA Nsight Systems from [here](https://developer.nvidia.com/nsight-systems/get-started).


You can also get resources from [Open Hackathons technical resource page](https://www.openhackathons.org/s/technical-resources)


--- 

## Licensing 

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials may include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.